In [1]:
import pandas as pd
import pandas_gbq 
from datetime import datetime
import numpy as np
from google.cloud import bigquery



In [10]:
project_id = "infra-itaborai"

query = """
SELECT Numero_Guia, Valor_da_Nota
FROM `infra-itaborai.mds_ibrape.tb_notas`


"""

df_notas = pandas_gbq.read_gbq(query, project_id=project_id)


Downloading: 100%|██████████|


In [62]:
df_notas

,Numero_Guia,Valor_da_Nota
0,102348,14918.00
1,102348,34202.00
2,102348,200000.00
3,102348,12156.00
4,102342,161134.00
...,...,...
1530481,141523,19.64
1530482,141523,117.39
1530483,141521,700.00
1530484,141521,4900.00


In [63]:
df = pd.read_csv(r"C:\Users\Nelio\OneDrive\Villam\POWERBI\mds_Notas\Notas\Dados usados\Guias\19-02-2026_Guias_Geral.csv",  sep=";")

In [64]:
df.columns = (
    df.columns
    .str.strip()               
    .str.lower()                
    .str.replace(" ", "_")     
    .str.replace(r"[^\w]", "", regex=True)  
)

conditions = [
    df["tipo"] == "ME",
    df["tipo"] == "Retido",
    df["tipo"] == "Direto"
]

choices = [
    "Prestador",
    "Tomador",
    "Empresa de Fora"
]

df["quem_paga"] = np.select(conditions, choices, default="OUTRO STATUS")


In [65]:
df_guias = df

In [ ]:
df_guias["pagamento"] = pd.to_datetime(df_guias["pagamento"], errors="coerce", dayfirst=True)
df_guias["vencimento"] = pd.to_datetime(df_guias["vencimento"], errors="coerce", dayfirst=True)

In [67]:
df_guias

,emissor,razão_social,prestador,razão_social1,vencimento,vlr_dam,dam,pagamento,tipo,cancelada,quem_paga
0,00.913.096/0001-89,BABY CLINICA LTDA,NaN,NaN,2022-09-15,"29,43",104775,2022-09-15,ME,NÃO,Prestador
1,47.960.950/1399-87,MAGAZINE LUIZA S/A,NaN,NaN,2022-08-31,"8,77",104487,NaT,ME,SIM,Prestador
2,47.960.950/1399-87,MAGAZINE LUIZA S/A,NaN,NaN,2022-08-31,"0,65",104488,2022-08-29,ME,NÃO,Prestador
3,09.492.725/0001-19,FIRE SERVICE DO BRASIL LTDA ME,NaN,NaN,2022-09-15,736,104555,NaT,ME,NÃO,Prestador
4,09.492.725/0001-19,FIRE SERVICE DO BRASIL LTDA ME,NaN,NaN,2022-09-15,465,104556,NaT,ME,NÃO,Prestador
...,...,...,...,...,...,...,...,...,...,...,...
41351,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,2026-02-16,"45,1",141306,NaT,Direto,NÃO,Empresa de Fora
41352,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,2026-02-04,"47,6",141304,NaT,Direto,NÃO,Empresa de Fora
41353,62.294.600/0001-67,JACKELINE ARAGAO - PSICOLOGIA & EMOCOES LTDA,62.294.600/0001-67,JACKELINE ARAGAO - PSICOLOGIA & EMOCOES LTDA,2026-01-29,"358,6",141199,NaT,Direto,NÃO,Empresa de Fora
41354,72.934.748/0001-72,MAGIC GAMES EMPREENDIMENTOS COMERCIAIS LTDA,72.934.748/0001-72,MAGIC GAMES EMPREENDIMENTOS COMERCIAIS LTDA,2026-02-16,"1.830,35",141672,NaT,Direto,NÃO,Empresa de Fora


In [ ]:
import pandas as pd
from datetime import datetime


hoje = pd.Timestamp(datetime.today().date())

# Cria coluna padrão
df_guias["sttsGuia"] = "OUTRO STATUS"


df_guias["existe_nota"] = df_guias["dam"].isin(df_notas["Numero_Guia"])


df_guias.loc[df_guias["cancelada"] == "SIM", "sttsGuia"] = "Guia Cancelada"

df_guias.loc[(~df_guias["existe_nota"]) & (df_guias["vencimento"] < hoje), "sttsGuia"] = "Guia Avulsa Vencida"

df_guias.loc[(~df_guias["existe_nota"]) & (df_guias["vencimento"] >= hoje), "sttsGuia"] = "Guia Avulsa a Vencer"

df_guias.loc[df_guias["pagamento"].notna(), "sttsGuia"] = "Guia PAGA"

df_guias.loc[(df_guias["vencimento"].notna()) & (df_guias["pagamento"].isna()) & (df_guias["vencimento"] < hoje), "sttsGuia"] = "Guia Vencida"

df_guias.loc[(df_guias["vencimento"].notna()) & (df_guias["pagamento"].isna()) & (df_guias["vencimento"] >= hoje), "sttsGuia"] = "Guia a Vencer"

In [69]:
df_guias

,emissor,razão_social,prestador,razão_social1,vencimento,vlr_dam,dam,pagamento,tipo,cancelada,quem_paga,sttsGuia,existe_nota
0,00.913.096/0001-89,BABY CLINICA LTDA,NaN,NaN,2022-09-15,"29,43",104775,2022-09-15,ME,NÃO,Prestador,Guia PAGA,False
1,47.960.950/1399-87,MAGAZINE LUIZA S/A,NaN,NaN,2022-08-31,"8,77",104487,NaT,ME,SIM,Prestador,Guia Vencida,False
2,47.960.950/1399-87,MAGAZINE LUIZA S/A,NaN,NaN,2022-08-31,"0,65",104488,2022-08-29,ME,NÃO,Prestador,Guia PAGA,False
3,09.492.725/0001-19,FIRE SERVICE DO BRASIL LTDA ME,NaN,NaN,2022-09-15,736,104555,NaT,ME,NÃO,Prestador,Guia Vencida,False
4,09.492.725/0001-19,FIRE SERVICE DO BRASIL LTDA ME,NaN,NaN,2022-09-15,465,104556,NaT,ME,NÃO,Prestador,Guia Vencida,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
41351,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,2026-02-16,"45,1",141306,NaT,Direto,NÃO,Empresa de Fora,Guia Vencida,False
41352,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,59.187.570/0001-85,BRASIL EXPRESS LOGISTICA LTDA,2026-02-04,"47,6",141304,NaT,Direto,NÃO,Empresa de Fora,Guia Vencida,False
41353,62.294.600/0001-67,JACKELINE ARAGAO - PSICOLOGIA & EMOCOES LTDA,62.294.600/0001-67,JACKELINE ARAGAO - PSICOLOGIA & EMOCOES LTDA,2026-01-29,"358,6",141199,NaT,Direto,NÃO,Empresa de Fora,Guia Vencida,False
41354,72.934.748/0001-72,MAGIC GAMES EMPREENDIMENTOS COMERCIAIS LTDA,72.934.748/0001-72,MAGIC GAMES EMPREENDIMENTOS COMERCIAIS LTDA,2026-02-16,"1.830,35",141672,NaT,Direto,NÃO,Empresa de Fora,Guia Vencida,False


In [ ]:
project_id = "infra-itaborai"         
dataset_table = "mds_ibrape.fGuias" 


to_gbq(df_guias, dataset_table, project_id=project_id, if_exists="replace")

100%|██████████| 1/1 [00:00<00:00, 17924.38it/s]


In [3]:
project_id = "infra-itaborai"

query = """
 WITH rfb AS (
        SELECT  
        *
       FROM infra-itaborai.mds_cnpj.dbt_cnpj_itaborai_ativas
    )
SELECT * FROM RFB

"""

rfb = pandas_gbq.read_gbq(query, project_id=project_id)

Downloading: 100%|██████████|


In [4]:
rfb

,CNPJ_RAZAO_SOCIAL_CNAE_PRINCIPAL,CNPJ_RAZAO,CNPJ_COMPLETO,RAZAO_SOCIAL,CNPJ_BASICO,CNPJ_ORDEM,CNPJ_DV,MATRIZ_FILIAL,NOME_FANTASIA,SITUACAO_CADASTRAL,...,ENTE_RESPONSAVEL,OPCAO_PELO_SIMPLES,DATA_OPCAO_SIMPLES,DATA_EXCLUSAO_DO_SIMPLES,OPCAO_PELO_MEI,DATA_OPCAO_MEI,DATA_EXCLUSAO_DO_MEI,CNAE,DESCRICAO_CNAE,CLASIFICACAO_PRINCIPAL
0,57461698000160 - RANCHO MJ AGRO E AVES LTDA - ...,57461698000160 - RANCHO MJ AGRO E AVES LTDA,57461698000160,RANCHO MJ AGRO E AVES LTDA,57461698,0001,60,1,RANCHO MJ,02,...,None,N,2025-01-01,2025-12-31,N,NaT,NaT,None,None,None
1,62539496000123 - AGRIBELLA AGRO E COMERCIO LTD...,62539496000123 - AGRIBELLA AGRO E COMERCIO LTDA,62539496000123,AGRIBELLA AGRO E COMERCIO LTDA,62539496,0001,23,1,None,02,...,None,None,NaT,NaT,None,NaT,NaT,None,None,None
2,36050727000120 - PAULO ROBERTO AVELAR 72105240...,36050727000120 - PAULO ROBERTO AVELAR 72105240704,36050727000120,PAULO ROBERTO AVELAR 72105240704,36050727,0001,20,1,None,02,...,None,S,2020-01-18,NaT,S,2020-01-18,NaT,None,None,None
3,14223332000140 - 14.223.332 CARMEN MELLO DA SI...,14223332000140 - 14.223.332 CARMEN MELLO DA SILVA,14223332000140,14.223.332 CARMEN MELLO DA SILVA,14223332,0001,40,1,None,02,...,None,S,2025-01-01,NaT,S,2025-01-01,NaT,None,None,None
4,10799959000192 - FAZENDA E HARAS DO REI LTDA -...,10799959000192 - FAZENDA E HARAS DO REI LTDA,10799959000192,FAZENDA E HARAS DO REI LTDA,10799959,0001,92,1,HARAS DO REI,02,...,None,None,NaT,NaT,None,NaT,NaT,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32691,45421074000130 - MARINA DIAS DOS SANTOS 075927...,45421074000130 - MARINA DIAS DOS SANTOS 075927...,45421074000130,MARINA DIAS DOS SANTOS 07592725738,45421074,0001,30,1,None,04,...,None,S,2022-02-23,NaT,S,2022-02-23,NaT,9700500,SERVIÇOS DOMÉSTICOS,SERVIÇO
32692,48615391000185 - SARAH LORRANA GONCALVES BICAR...,48615391000185 - SARAH LORRANA GONCALVES BICAR...,48615391000185,SARAH LORRANA GONCALVES BICARIOS DE AZEREDO 19...,48615391,0001,85,1,None,04,...,None,S,2022-11-15,NaT,S,2022-11-15,NaT,9700500,SERVIÇOS DOMÉSTICOS,SERVIÇO
32693,44789555000130 - LUIZ ANTONIO VERISSIMO 998734...,44789555000130 - LUIZ ANTONIO VERISSIMO 998734...,44789555000130,LUIZ ANTONIO VERISSIMO 99873478787,44789555,0001,30,1,None,04,...,None,S,2022-01-08,NaT,S,2022-01-08,NaT,9700500,SERVIÇOS DOMÉSTICOS,SERVIÇO
32694,40941028000148 - TATIANA JORDAO DOS SANTOS 059...,40941028000148 - TATIANA JORDAO DOS SANTOS 059...,40941028000148,TATIANA JORDAO DOS SANTOS 05971508776,40941028,0001,48,1,None,04,...,None,S,2021-02-22,NaT,S,2021-02-22,NaT,9700500,SERVIÇOS DOMÉSTICOS,SERVIÇO
